# DSC259 Final Project: Power Outages

**Name(s)**: Randall "Scotty" Rogers, Jillian O'Neel, Hans Hanson

**Website Link**: https://scotty-ucsd.github.io/dsc259_final_project_rsrogers/

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc259r_utils import * # Feel free to uncomment and use this.

## Table of Contents

- [Step 1: Introduction](#step1)
- [Step 2: Data Cleaning and Exploratory Analysis](#step2)
    - [Load Data](#load_data)
    - [Initial Inspection](#initial_inspection)
    - [Data Cleaning](#data_cleaning)
    - [Univariate Analysis](#univariate_analysis)
    - [Bivariate Analysis](#bivariate_analysis)
    - [Interesting Aggregates](#interesting_aggregates)
- [Step 3: Assessment of Missingness](#step3)
- [Step 4: Hypothesis Testing](#step4)
- [Step 5: Framing a Prediction Problem](#step5)
- [Step 6: Baseline Model](#step6)
- [Step 7: Final Model](#step7)
- [Step 8: Fairness Analysis](#step8)

<a id='step1'></a>
## Step 1: Introduction

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

In [19]:
# TODO


<a id='step2'></a>
## Step 2: Data Cleaning and Exploratory Data Analysis

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

<a id='load_data'></a>
#### Load Data

#DO WE NEED TO MAKE IT DOWNLOAD THE RAW EXCEL FILE FROM PURDUE? I have a notebook that cleans it from that excel file
https://engineering.purdue.edu/LASCI/research-data/outages

In [20]:
data = "data/outage_clean.csv"
df = pd.read_csv(data)

<a id='initial_inspection'></a>
#### Initial Inspection

##### Check Number of Columns and Rows

In [21]:
label_w = 30
num_w = 8
print(f"{'columns in outage data:':>{label_w}} {df.shape[1]:>{num_w}}")
print(f"{'rows in outage data:':>{label_w}} {df.shape[0]:>{num_w}}")
print(f"{'total elements in outage data:':>{label_w}} {df.size:>{num_w}}")

       columns in outage data:       56
          rows in outage data:     1534
total elements in outage data:    85904


##### Check Nulls and Datatypes

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1534 entries, 0 to 1533
Data columns (total 56 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   OBS                      1534 non-null   int64  
 1   YEAR                     1534 non-null   int64  
 2   MONTH                    1525 non-null   float64
 3   U.S._STATE               1534 non-null   str    
 4   POSTAL.CODE              1534 non-null   str    
 5   NERC.REGION              1534 non-null   str    
 6   CLIMATE.REGION           1528 non-null   str    
 7   ANOMALY.LEVEL            1525 non-null   float64
 8   CLIMATE.CATEGORY         1525 non-null   str    
 9   OUTAGE.START.DATE        1525 non-null   str    
 10  OUTAGE.START.TIME        1525 non-null   str    
 11  OUTAGE.RESTORATION.DATE  1476 non-null   str    
 12  OUTAGE.RESTORATION.TIME  1476 non-null   str    
 13  CAUSE.CATEGORY           1534 non-null   str    
 14  CAUSE.CATEGORY.DETAIL    1063 non-n

##### Check Date and Time Columns

In [23]:
# check datetime columns
date_time = [i for i in df.columns if 'TIME' in i] +[i for i in df.columns if 'DATE' in i]
df[date_time].head()

,OUTAGE.START.TIME,OUTAGE.RESTORATION.TIME,OUTAGE.START.DATE,OUTAGE.RESTORATION.DATE
0,5:00:00 PM,8:00:00 PM,"Friday, July 1, 2011","Sunday, July 3, 2011"
1,6:38:00 PM,6:39:00 PM,"Sunday, May 11, 2014","Sunday, May 11, 2014"
2,8:00:00 PM,10:00:00 PM,"Tuesday, October 26, 2010","Thursday, October 28, 2010"
3,4:30:00 AM,11:00:00 PM,"Tuesday, June 19, 2012","Wednesday, June 20, 2012"
4,2:00:00 AM,7:00:00 AM,"Saturday, July 18, 2015","Sunday, July 19, 2015"


<a id='data_cleaning'></a>
#### Data Cleaning

In [24]:
# Combine the columns with a space separating date and time
start_combined = df['OUTAGE.START.DATE'] + ' ' + df['OUTAGE.START.TIME']
restoration_combined = df['OUTAGE.RESTORATION.DATE'] + ' ' + df['OUTAGE.RESTORATION.TIME']

# Define the explicit format string matching your dataframe's text
# %A: Full Weekday, %B: Full Month, %d: Day, %Y: 4-digit Year
# %I: Hour (12hr), %M: Minute, %S: Second, %p: AM/PM
date_format = "%A, %B %d, %Y %I:%M:%S %p"

# Apply to_datetime with the explicitly specified format
df['START.DATETIME'] = pd.to_datetime(start_combined, format=date_format, errors='coerce')
df['RESTORATION.DATETIME'] = pd.to_datetime(restoration_combined, format=date_format, errors='coerce')
# 3. Check the results
print(df[['START.DATETIME', 'RESTORATION.DATETIME']].info())
print(df[['START.DATETIME', 'RESTORATION.DATETIME']].head())

<class 'pandas.DataFrame'>
RangeIndex: 1534 entries, 0 to 1533
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   START.DATETIME        1525 non-null   datetime64[us]
 1   RESTORATION.DATETIME  1476 non-null   datetime64[us]
dtypes: datetime64[us](2)
memory usage: 24.1 KB
None
       START.DATETIME RESTORATION.DATETIME
0 2011-07-01 17:00:00  2011-07-03 20:00:00
1 2014-05-11 18:38:00  2014-05-11 18:39:00
2 2010-10-26 20:00:00  2010-10-28 22:00:00
3 2012-06-19 04:30:00  2012-06-20 23:00:00
4 2015-07-18 02:00:00  2015-07-19 07:00:00


In [25]:
# 2. Drop the redundant string columns to clean up the dataframe
cols_to_drop = ['OUTAGE.START.DATE', 'OUTAGE.START.TIME', 
                'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME']
df_cleaned = df.drop(columns=cols_to_drop)

# 3. Filter out rows where crucial datetime info is missing for our analysis
df_cleaned = df_cleaned.dropna(subset=['START.DATETIME', 'RESTORATION.DATETIME', 'OUTAGE.DURATION'])

# Show the head of the cleaned DataFrame (Assignment Requirement)
display(df_cleaned.head())

,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,CAUSE.CATEGORY,...,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,START.DATETIME,RESTORATION.DATETIME
0,1,2011,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2011-07-01 17:00:00,2011-07-03 20:00:00
1,2,2014,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,intentional attack,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2014-05-11 18:38:00,2014-05-11 18:39:00
2,3,2010,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2010-10-26 20:00:00,2010-10-28 22:00:00
3,4,2012,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2012-06-19 04:30:00,2012-06-20 23:00:00
4,5,2015,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,severe weather,...,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743,2015-07-18 02:00:00,2015-07-19 07:00:00


In [36]:
df_cleaned.shape

(1476, 54)

<a id='univariate_analysis'></a>
#### Univariate Analysis

In [26]:
# Plot 1: Distribution of Outage Duration
fig_uni_1 = px.histogram(df_cleaned, x='OUTAGE.DURATION', nbins=50, 
                         title='Distribution of Outage Durations (Minutes)',
                         labels={'OUTAGE.DURATION': 'Outage Duration (Mins)'})
fig_uni_1.write_html("outage_duration.html", include_plotlyjs='cdn')
fig_uni_1.show()

# Plot 2: Distribution of Outage Causes
fig_uni_2 = px.bar(df_cleaned['CAUSE.CATEGORY'].value_counts().reset_index(), 
                   x='CAUSE.CATEGORY', y='count', 
                   title='Count of Outages by Cause Category')
fig_uni_2.write_html("cause_category.html", include_plotlyjs='cdn')
fig_uni_2.show()

<a id='bivariate_analysis'></a>
#### Bivariate Analysis

In [27]:
df['START.MONTH'] = df['START.DATETIME'].dt.month_name()

fig_bi_1 = px.box(df, 
                  x='START.MONTH', 
                  y='OUTAGE.DURATION',
                  title='Distribution of Outage Duration by Month',
                  category_orders={"START.MONTH": ["January", "February", "March", "April", "May", "June", 
                                                   "July", "August", "September", "October", "November", "December"]},
                  labels={'START.MONTH': 'Month', 'OUTAGE.DURATION': 'Outage Duration (Minutes)'})

fig_bi_1.update_yaxes(type="log")
fig_bi_1.write_html("duration_month.html", include_plotlyjs='cdn')
fig_bi_1.show()


fig_bi_1.write_image("duration_month.png", validate=True, width=800, height=600, scale=1)

<a id='interesting_aggregates'></a>
#### Interesting Aggregates

In [28]:
# Group by Climate Region and Cause, looking at median duration and average customers affected
agg_table = df_cleaned.groupby(['CLIMATE.REGION', 'CAUSE.CATEGORY']).agg({
    'OUTAGE.DURATION': 'median',
    'CUSTOMERS.AFFECTED': 'mean'
}).reset_index().round(2)

# Pivot the table for better readability
pivot_table = agg_table.pivot(index='CLIMATE.REGION', columns='CAUSE.CATEGORY', values='OUTAGE.DURATION')
display(pivot_table)

CAUSE.CATEGORY,equipment failure,fuel supply emergency,intentional attack,islanding,public appeal,severe weather,system operability disruption
CLIMATE.REGION,,,,,,,
Central,149.0,7500.5,50.0,96.0,1410.0,1680.0,65.0
East North Central,761.0,13564.0,648.5,1.0,733.0,4005.0,2694.0
Northeast,159.0,12240.0,1.0,881.0,2760.0,3189.0,234.5
Northwest,702.0,1.0,74.0,21.0,898.0,3507.0,157.5
South,227.0,20160.0,100.0,493.5,430.0,2027.5,373.0
Southeast,308.5,NaN,108.0,NaN,4320.0,1355.0,110.0
Southwest,35.0,76.0,56.0,2.0,2275.0,2425.0,284.0
West,269.0,882.5,108.0,128.5,420.0,962.0,199.0
West North Central,61.0,NaN,0.5,56.0,439.5,83.0,NaN


<a id='step3'></a>
## Step 3: Assessment of Missingness

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

In [29]:
# TODO

<a id='step4'></a>
## Step 4: Hypothesis Testing

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

In [30]:
# TODO

<a id='step5'></a>
## Step 5: Framing a Prediction Problem

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

In [31]:
# TODO

<a id='step6'></a>
## Step 6: Baseline Model

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

In [32]:
# TODO

<a id='step7'></a>
## Step 7: Final Model

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

In [33]:
# TODO

<a id='step8'></a>
## Step 8: Fairness Analysis

[canvas link to instructions](https://canvas.ucsd.edu/courses/71325/pages/about-the-final-project)

In [34]:
# TODO